In [1]:
from pathlib import Path
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive

# client_secrets.json lives in the repo root, not next to this notebook
secrets = next(p / "client_secrets.json" for p in [Path.cwd(), *Path.cwd().parents] if (p / "client_secrets.json").exists())

gauth = GoogleAuth()
gauth.settings["client_config_file"] = str(secrets)
gauth.settings["get_refresh_token"] = True  # lets the saved login be renewed silently
creds_file = str(secrets.parent / "credentials.json")

gauth.LoadCredentialsFile(creds_file)  # reuse saved login if present
if gauth.credentials is None:
    gauth.LocalWebserverAuth()  # first time only: opens browser
elif gauth.access_token_expired:
    gauth.Refresh()
else:
    gauth.Authorize()
gauth.SaveCredentialsFile(creds_file)

drive = GoogleDrive(gauth)

file_list = drive.ListFile({
    "q": "trashed=false"
}).GetList()

for file in file_list:
    print(file["title"], file["id"], file["mimeType"])

c:\Users\stand\cockroach-survivor-learn\.venv\Lib\site-packages\oauth2client\_helpers.py:255: UserWarning: Cannot access c:\Users\stand\cockroach-survivor-learn\credentials.json: No such file or directory
  warnings.warn(_MISSING_FILE_MESSAGE.format(filename))


Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?client_id=830928765974-2clbj0rgf2t1j17uv2bo5ag4vd4kshut.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive&access_type=offline&response_type=code&approval_prompt=force

Authentication successful.
39_Harnessing_the_Universal_Geometry_of_Embeddings.pdf 1DnKBIq96S6wmBEnkJVislEQkwJ9dVlnL application/pdf
RAG-data 1w-XQ5cFqB2ruY0fVBHL92RSq97u6nlcS application/vnd.google-apps.folder
LangChain_LangGraph_Study_Guide.docx 1kSuwLoJMqG3zprf1jMKcG14NIP2TUSvz application/vnd.openxmlformats-officedocument.wordprocessingml.document
Cockroach-TheFrontDeskThatNeverSleep 19xhY8qOpdPis0oe55pMJ_ONkwTIW--af application/vnd.google-apps.folder
Reflexion: Language Agents with Verbal Reinforcement Learning 1XGL8TixKAQT300e_dq0-50IO1-Jw2NyL application/pdf
Langchain 14P0fOien_Gq_AMK5YbXHJPvBRJ0KemZ6 application/vnd.google-apps.folder
origin 1_fgep0wz7tG

In [2]:
folders = drive.ListFile({
    "q": (
        "title = 'Rag-data' "
        "and mimeType = 'application/vnd.google-apps.folder' "
        "and trashed = false"
    )
}).GetList()

for folder in folders:
    print("Name:", folder["title"])
    print("Folder ID:", folder["id"])

Name: RAG-data
Folder ID: 1w-XQ5cFqB2ruY0fVBHL92RSq97u6nlcS


In [3]:
folder_id = "1w-XQ5cFqB2ruY0fVBHL92RSq97u6nlcS"

files = drive.ListFile({
    "q": f"'{folder_id}' in parents and trashed = false"
}).GetList()

for file in files:
    print(file["title"], "|", file["mimeType"])

Langchain | application/vnd.google-apps.folder
FastAPI | application/vnd.google-apps.folder
Research Paper | application/vnd.google-apps.folder


In [4]:
research_folder = next(
    file for file in files
    if file["title"] == "Research Paper"
)

research_folder_id = research_folder["id"]

research_files = drive.ListFile({
    "q": f"'{research_folder_id}' in parents and trashed = false"
}).GetList()

for file in research_files:
    print(file["title"], "|", file["mimeType"])

39_Harnessing_the_Universal_Geometry_of_Embeddings.pdf | application/pdf
how_to_read_llm_papers.md | text/markdown
index.csv | text/csv
38_AI_Co-Mathematician.pdf | application/pdf
37_Cola_DLM.pdf | application/pdf
36_How_Controllable_Are_Large_Language_Models_SteerEval.pdf | application/pdf
35_Evaluating_Language_Models_for_Harmful_Manipulation.pdf | application/pdf
34_Reverse_CAPTCHA.pdf | application/pdf
33_MAS-FIRE.pdf | application/pdf
32_FinRetrieval.pdf | application/pdf
31_Try_Check_and_Retry.pdf | application/pdf
30_Gemini_2.5.pdf | application/pdf
29_Qwen3_Technical_Report.pdf | application/pdf
28_Why_Do_Multi-Agent_LLM_Systems_Fail.pdf | application/pdf
27_Does_RL_Really_Incentivize_Reasoning_Capacity_Beyond_the_Base_Model.pdf | application/pdf
26_DAPO.pdf | application/pdf
25_Alignment_Faking_in_Large_Language_Models.pdf | application/pdf
24_DeepSeek-V3.pdf | application/pdf
23_Scaling_LLM_Test-Time_Compute_Optimally.pdf | application/pdf
22_DeepSeekMath_GRPO.pdf | applicat

In [5]:
from pathlib import Path

# Find the paper in the list we already retrieved
paper = next(
    file for file in research_files
    if file["title"] == "03_Attention_Is_All_You_Need.pdf"
)

# Create a local folder for downloaded documents
download_dir = Path.cwd() / "downloaded_papers"
download_dir.mkdir(parents=True, exist_ok=True)

# Choose where to save the PDF
pdf_path = download_dir / paper["title"]

# Download the file
paper.GetContentFile(str(pdf_path))

print("Downloaded:", pdf_path)
print("File size:", pdf_path.stat().st_size, "bytes")

Downloaded: c:\Users\stand\cockroach-survivor-learn\connect_to_google_drive\downloaded_papers\03_Attention_Is_All_You_Need.pdf
File size: 2215244 bytes


In [6]:
%pip install pypdf

Note: you may need to restart the kernel to use updated packages.


In [7]:
from pypdf import PdfReader

reader = PdfReader(str(pdf_path))

print("Total pages:", len(reader.pages))

first_page = reader.pages[0]
text = first_page.extract_text()

print("\nExtracted characters:", len(text or ""))
print("\nFirst 1500 characters:\n")
print((text or "")[:1500])

Total pages: 15

Extracted characters: 2857

First 1500 characters:

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention me

In [8]:
%pip install langchain-core langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [9]:
from langchain_core.documents import Document

documents = []

for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""

    if not text.strip():
        continue

    document = Document(
        page_content=text,
        metadata={
            "source": pdf_path.name,
            "page": page_number,
        },
    )

    documents.append(document)

print("Pages with extracted text:", len(documents))
print("First document metadata:", documents[0].metadata)
print("First document characters:", len(documents[0].page_content))

Pages with extracted text: 15
First document metadata: {'source': '03_Attention_Is_All_You_Need.pdf', 'page': 1}
First document characters: 2857


In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_documents(documents)

print("Original page documents:", len(documents))
print("Total chunks:", len(chunks))

print("\nFirst chunk metadata:")
print(chunks[0].metadata)

print("\nFirst chunk text:")
print(chunks[0].page_content[:1000])

Original page documents: 15
Total chunks: 52

First chunk metadata:
{'source': '03_Attention_Is_All_You_Need.pdf', 'page': 1}

First chunk text:
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a ne

In [11]:
from pypdf import PdfReader
from langchain_core.documents import Document

all_documents = []
failed_files = []

pdf_files = [
    file for file in research_files
    if file["mimeType"] == "application/pdf"
]

for file in pdf_files:
    filename = file["title"]
    local_path = download_dir / filename

    try:
        # Download only if not already present
        if not local_path.exists():
            file.GetContentFile(str(local_path))

        # Extract text page by page
        reader = PdfReader(str(local_path))

        for page_number, page in enumerate(reader.pages, start=1):
            text = page.extract_text() or ""

            if not text.strip():
                continue

            all_documents.append(
                Document(
                    page_content=text,
                    metadata={
                        "source": filename,
                        "page": page_number,
                        "drive_file_id": file["id"],
                    },
                )
            )

        print(f"Processed: {filename}")

    except Exception as error:
        failed_files.append((filename, str(error)))
        print(f"FAILED: {filename} — {error}")

# Chunk all extracted pages
all_chunks = splitter.split_documents(all_documents)

print("\n--- INGESTION SUMMARY ---")
print("PDFs found:", len(pdf_files))
print("Pages with text:", len(all_documents))
print("Total chunks:", len(all_chunks))
print("Failed files:", len(failed_files))

for filename, error in failed_files:
    print("ERROR:", filename, error)

Processed: 39_Harnessing_the_Universal_Geometry_of_Embeddings.pdf
Processed: 38_AI_Co-Mathematician.pdf


Exceeded 5000 form XObject invocations while extracting text; further form content is skipped.


Processed: 37_Cola_DLM.pdf
Processed: 36_How_Controllable_Are_Large_Language_Models_SteerEval.pdf
Processed: 35_Evaluating_Language_Models_for_Harmful_Manipulation.pdf
Processed: 34_Reverse_CAPTCHA.pdf
Processed: 33_MAS-FIRE.pdf
Processed: 32_FinRetrieval.pdf
Processed: 31_Try_Check_and_Retry.pdf
Processed: 30_Gemini_2.5.pdf
Processed: 29_Qwen3_Technical_Report.pdf
Processed: 28_Why_Do_Multi-Agent_LLM_Systems_Fail.pdf
Processed: 27_Does_RL_Really_Incentivize_Reasoning_Capacity_Beyond_the_Base_Model.pdf
Processed: 26_DAPO.pdf
Processed: 25_Alignment_Faking_in_Large_Language_Models.pdf
Processed: 24_DeepSeek-V3.pdf
Processed: 23_Scaling_LLM_Test-Time_Compute_Optimally.pdf
Processed: 22_DeepSeekMath_GRPO.pdf
Processed: 21_The_Llama_3_Herd_of_Models.pdf
Processed: 20_Lost_in_the_Middle.pdf
Processed: 19_Retrieval-Augmented_Generation.pdf
Processed: 18_Mixtral_of_Experts.pdf
Processed: 17_FlashAttention.pdf
Processed: 16_LoRA.pdf
Processed: 15_LLaMA.pdf
Processed: 14_DeepSeek-R1.pdf
Process

In [12]:
from collections import defaultdict

paper_stats = defaultdict(lambda: {
    "pages": 0,
    "characters": 0,
})

for doc in all_documents:
    source = doc.metadata["source"]
    paper_stats[source]["pages"] += 1
    paper_stats[source]["characters"] += len(doc.page_content)

for source, stats in sorted(paper_stats.items()):
    avg = stats["characters"] / stats["pages"]

    print(
        f"{source[:42]:42} "
        f"Pages: {stats['pages']:3} "
        f"Avg chars/page: {avg:7.0f}"
    )

    if avg < 500:
        print("  WARNING: Unusually low text extraction")

01_Word2Vec.pdf                            Pages:  12 Avg chars/page:    3230
02_Neural_Machine_Translation_by_Jointly_L Pages:  15 Avg chars/page:    3337
03_Attention_Is_All_You_Need.pdf           Pages:  15 Avg chars/page:    2633
04_BERT.pdf                                Pages:  16 Avg chars/page:    4008
05_GPT-3_Language_Models_are_Few-Shot_Lear Pages:  75 Avg chars/page:    3156
06_Scaling_Laws_for_Neural_Language_Models Pages:  30 Avg chars/page:    2949
07_Chinchilla_Training_Compute-Optimal_LLM Pages:  36 Avg chars/page:    2653
08_InstructGPT.pdf                         Pages:  68 Avg chars/page:    2673
09_Constitutional_AI.pdf                   Pages:  34 Avg chars/page:    3494
10_DPO_Direct_Preference_Optimization.pdf  Pages:  27 Avg chars/page:    3440
11_Chain-of-Thought_Prompting.pdf          Pages:  43 Avg chars/page:    3156
12_ReAct.pdf                               Pages:  33 Avg chars/page:    3336
13_Toolformer.pdf                          Pages:  17 Avg chars/

In [13]:
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
)

strategies = {
    "fixed_character": CharacterTextSplitter(
        separator="",
        chunk_size=1000,
        chunk_overlap=200,
    ),

    "recursive": RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
    ),

    "sentence_aware": RecursiveCharacterTextSplitter(
        separators=[". ", "? ", "! ", "\n\n", "\n", " ", ""],
        chunk_size=1000,
        chunk_overlap=200,
    ),
}

results = {}

for name, strategy in strategies.items():
    chunks_for_strategy = strategy.split_documents(all_documents)

    lengths = [
        len(chunk.page_content)
        for chunk in chunks_for_strategy
    ]

    results[name] = {
        "chunks": len(chunks_for_strategy),
        "avg_length": sum(lengths) / len(lengths),
        "min_length": min(lengths),
        "max_length": max(lengths),
    }

for name, stats in results.items():
    print(f"\n{name}")
    for metric, value in stats.items():
        print(f"  {metric}: {value:.1f}")


fixed_character
  chunks: 6319.0
  avg_length: 904.4
  min_length: 63.0
  max_length: 1000.0

recursive
  chunks: 6331.0
  avg_length: 872.0
  min_length: 63.0
  max_length: 1000.0

sentence_aware
  chunks: 6632.0
  avg_length: 781.2
  min_length: 12.0
  max_length: 1000.0


In [14]:
target_paper = "03_Attention_Is_All_You_Need.pdf"

sample_documents = [
    doc for doc in all_documents
    if doc.metadata["source"] == target_paper
]

for name, strategy in strategies.items():
    sample_chunks = strategy.split_documents(sample_documents)

    print(f"\n{'=' * 60}")
    print(f"STRATEGY: {name}")
    print(f"{'=' * 60}")

    for i, chunk in enumerate(sample_chunks[:3], start=1):
        print(f"\nChunk {i}")
        print("Page:", chunk.metadata["page"])
        print("Length:", len(chunk.page_content))
        print("START:", repr(chunk.page_content[:150]))
        print("END:  ", repr(chunk.page_content[-150:]))


STRATEGY: fixed_character

Chunk 1
Page: 1
Length: 1000
START: 'Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalis'
END:   ' a new simple network architecture, the Transformer,\nbased solely on attention mechanisms, dispensing with recurrence and convolutions\nentirely. Exper'

Chunk 2
Page: 1
Length: 1000
START: 'decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer,\nbased solely on attention mechanisms, dispensin'
END:   't successfully to English constituency parsing both with\nlarge and limited training data.\n∗Equal contribution. Listing order is random. Jakob proposed'

Chunk 3
Page: 1
Length: 1000
START: 'rmer generalizes well to\nother tasks by applying it successfully to English constituency parsing both with\nlarge and limited training data.\n∗Equal con'
END:   'z and Aidan spent countless long days designing various parts of 

In [15]:
improved_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=[
        "\n\n",
        "\n",
        ". ",
        "? ",
        "! ",
        " ",
        ""
    ],
    keep_separator="end",
)

improved_chunks = improved_splitter.split_documents(all_documents)

print("Original recursive chunks:", len(all_chunks))
print("Improved recursive chunks:", len(improved_chunks))

for chunk in improved_chunks[:3]:
    print("\nPage:", chunk.metadata["page"])
    print("Length:", len(chunk.page_content))
    print("START:", repr(chunk.page_content[:100]))
    print("END:", repr(chunk.page_content[-100:]))

Original recursive chunks: 6331
Improved recursive chunks: 6083

Page: 1
Length: 971
START: 'Harnessing the Universal Geometry of Embeddings\nRishi Jha Collin Zhang Vitaly Shmatikov John X. Morr'
END: 'itive information about\nunderlying documents, sufficient for classification and attribute inference.'

Page: 1
Length: 971
START: 'underlying documents, sufficient for classification and attribute inference.\nFigure 1: Left: input e'
END: 'ce, however,\ndifferent models encode texts into completely different and incompatible vector spaces.'

Page: 1
Length: 624
START: 'different models encode texts into completely different and incompatible vector spaces.\nThe Platonic'
END: 'ce on Neural Information Processing Systems (NeurIPS 2025).\narXiv:2505.12540v4  [cs.LG]  26 Jan 2026'


In [17]:
import json
from pathlib import Path

output_path = Path.cwd() / "research_chunks.jsonl"

with output_path.open("w", encoding="utf-8") as f:
    for i, chunk in enumerate(improved_chunks):
        record = {
            "chunk_id": f"chunk_{i:05d}",
            "text": chunk.page_content,
            "metadata": chunk.metadata,
        }

        f.write(json.dumps(record, ensure_ascii=True) + "\n")

print("Saved:", output_path)
print("Exported chunks:", len(improved_chunks))
print("File size:", output_path.stat().st_size, "bytes")

Saved: c:\Users\stand\cockroach-survivor-learn\connect_to_google_drive\research_chunks.jsonl
Exported chunks: 6083
File size: 6585343 bytes


In [18]:
with output_path.open("r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f]

print("Records loaded:", len(records))
print("First metadata:", records[0]["metadata"])

Records loaded: 6083
First metadata: {'source': '39_Harnessing_the_Universal_Geometry_of_Embeddings.pdf', 'page': 1, 'drive_file_id': '1DnKBIq96S6wmBEnkJVislEQkwJ9dVlnL'}


In [ ]:
# Google Docs -> export as plain text
doc = drive.CreateFile({"id": "1PxjgLTC9XGogl4694tiM8Zv3eUkXqWDWZbUfLw3IZ48"})
print(doc.GetContentString(mimetype="text/plain"))

# Markdown file -> read as-is
md = drive.CreateFile({"id": "1zCXQD0MRVAvo3QbIlDgc6gijCxePKfCe"})
print(md.GetContentString())